# workflow

> Main workflow class for structure decomposition

In [ ]:
#| default_exp workflow.workflow

In [ ]:
#| export
from typing import Dict, Any, List, Optional
from fasthtml.common import Div, P, A, Script, APIRouter
from fastcore.basics import patch

from cjm_fasthtml_interactions.patterns.step_flow import Step, StepFlow
from cjm_fasthtml_interactions.core.context import InteractionContext
from cjm_fasthtml_interactions.patterns.async_loading import AsyncLoadingContainer
from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

from cjm_fasthtml_design_system.text_tiers import text_tiers
from cjm_fasthtml_tailwind.utilities.spacing import m
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_plugin_system.core.manager import PluginManager
from cjm_plugin_system.core.queue import JobQueue
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_fasthtml_workflow_transcript_decomp.core.config import StructureDecompWorkflowConfig
from cjm_transcript_source_select.models import SelectionUrls
from cjm_transcript_segmentation.models import TextSegment
from cjm_transcript_vad_align.models import VADChunk
from cjm_transcript_source_select.services.source import SourceService
from cjm_transcript_review.services.graph import GraphService
from cjm_transcript_review.models import ReviewUrls
from cjm_transcript_review.components.review_card import AssembledSegment
from cjm_transcript_review.utils import generate_document_title
from cjm_transcript_verify.models import VerifyUrls
from cjm_transcript_verify.services.verify import VerifyService
from cjm_transcript_verify.components.step_renderer import render_verify_step

from cjm_fasthtml_step_progress.core.models import StepInfo
from cjm_fasthtml_step_progress.components.progress_bar import render_step_progress

# Step renderers
from cjm_transcript_source_select.components.step_renderer import render_selection_step
from cjm_transcript_review.components.step_renderer import render_review_step

## Session State Store Adapter

Bridges the pure `SQLiteWorkflowStateStore` (which accepts `session_id: str`)
to the `WorkflowStateStore` protocol expected by `StepFlow` (which passes `sess` objects).

In [ ]:
#| export
class _SessionStateStoreAdapter:
    """Adapter bridging StepFlow's sess-based protocol to the pure session_id-based store."""
    
    def __init__(
        self,
        store: SQLiteWorkflowStateStore  # The pure, framework-agnostic store
    ):
        """Wrap a pure store with session resolution."""
        self._store = store
    
    def get_current_step(
        self,
        flow_id: str,  # Workflow identifier
        sess: Any  # FastHTML session object
    ) -> Optional[str]:  # Current step ID or None
        """Get current step ID, resolving session from sess."""
        return self._store.get_current_step(flow_id, get_session_id(sess))
    
    def set_current_step(
        self,
        flow_id: str,  # Workflow identifier
        sess: Any,  # FastHTML session object
        step_id: str  # Step ID to set as current
    ) -> None:
        """Set current step ID, resolving session from sess."""
        self._store.set_current_step(flow_id, get_session_id(sess), step_id)
    
    def get_state(
        self,
        flow_id: str,  # Workflow identifier
        sess: Any  # FastHTML session object
    ) -> Dict[str, Any]:  # Workflow state dictionary
        """Get all workflow state, resolving session from sess."""
        return self._store.get_state(flow_id, get_session_id(sess))
    
    def update_state(
        self,
        flow_id: str,  # Workflow identifier
        sess: Any,  # FastHTML session object
        updates: Dict[str, Any]  # State updates to apply
    ) -> None:
        """Update workflow state, resolving session from sess."""
        self._store.update_state(flow_id, get_session_id(sess), updates)
    
    def clear_state(
        self,
        flow_id: str,  # Workflow identifier
        sess: Any  # FastHTML session object
    ) -> None:
        """Clear all workflow state, resolving session from sess."""
        self._store.clear_state(flow_id, get_session_id(sess))

## StructureDecompWorkflow

Main workflow class that orchestrates the structure decomposition process. It creates and manages:

- **SQLiteWorkflowStateStore**: Persistent state storage for workflow progress
- **Services**: Source, Segmentation, Alignment, and Graph services
- **StepFlow**: 3-step wizard with navigation
- **Routes**: API endpoints for the workflow

In [ ]:
#| export
class StructureDecompWorkflow:
    """Self-contained structure decomposition workflow."""
    
    # Class-level metadata for discovery
    workflow_name: str = "structure_decomposition"
    workflow_version: str = "1.0.0"
    workflow_category: str = "ingestion"
    workflow_title: str = "Structure Decomposition"
    workflow_description: str = "Decompose transcripts into traversable context graph spines"
    
    def __init__(
        self,
        plugin_manager: PluginManager,  # Plugin manager from host application
        config: Optional[StructureDecompWorkflowConfig] = None,  # Workflow configuration
        on_complete_redirect_url: Optional[str] = None,  # If set, "Start New Workflow" redirects here instead of resetting in place
    ):
        """Initialize the workflow with injected PluginManager."""
        self.config = config or StructureDecompWorkflowConfig()
        self._app = None  # Set in setup()
        
        # Where the "Start New Workflow" button should take the user on completion.
        # None means "reset in place" (the default, preserving existing behavior).
        # Hosts that wire up session management pass the session manager page URL here.
        # Read dynamically by `on_complete` so hosts can also set it after __init__.
        self._on_complete_redirect_url = on_complete_redirect_url
        
        # Store the injected PluginManager
        self._plugin_manager = plugin_manager
        
        # Create host-owned JobQueue for plugin job execution
        self._job_queue = JobQueue(plugin_manager)
        
        # Create SQLite-backed state store for persistence (pure, session_id-based)
        self._state_store = SQLiteWorkflowStateStore(
            db_path=self.config.get_state_db_path()
        )
        
        # Adapter for StepFlow compatibility (resolves sess -> session_id)
        self._session_adapter = _SessionStateStoreAdapter(self._state_store)
        
        # Create services (only those not managed by page-centric libraries)
        self._source_service = SourceService(
            plugin_manager,
            source_categories=self.config.source_categories
        )
        self._graph_service = GraphService(
            plugin_manager,
            plugin_name=self.config.graph_plugin
        )
        self._verify_service = VerifyService(
            plugin_manager,
            plugin_name=self.config.graph_plugin
        )
        
        # Create routers first (sets _sa_result needed by StepFlow)
        self._routers = self._create_routers()
        
        # Create StepFlow (uses _sa_result from router init)
        self._step_flow = self._create_step_flow()
        self._stepflow_router = self._step_flow.create_router(
            prefix=self.config.get_full_stepflow_prefix()
        )
    
    @classmethod
    def create_and_setup(
        cls,
        app,  # FastHTML application instance
        plugin_manager: PluginManager,  # Plugin manager from host application
        config: Optional[StructureDecompWorkflowConfig] = None,  # Workflow configuration
        on_complete_redirect_url: Optional[str] = None,  # Optional "Start New Workflow" redirect URL (see __init__)
    ) -> "StructureDecompWorkflow":  # Configured and setup workflow instance
        """Create, configure, and setup a workflow in one call."""
        workflow = cls(
            plugin_manager=plugin_manager,
            config=config,
            on_complete_redirect_url=on_complete_redirect_url,
        )
        workflow.setup(app)
        return workflow
    
    @property
    def on_complete_redirect_url(self) -> Optional[str]:  # Current redirect URL or None
        """The URL `on_complete` will redirect to on Phase 4 completion, or None to reset in place.
        
        Settable at runtime — `on_complete` reads this attribute dynamically at call time,
        so hosts can wire session management after workflow construction and still have
        the redirect take effect on the next "Start New Workflow" click.
        """
        return self._on_complete_redirect_url
    
    @on_complete_redirect_url.setter
    def on_complete_redirect_url(self, value: Optional[str]) -> None:
        self._on_complete_redirect_url = value
    
    @property
    def plugin_manager(self) -> PluginManager:  # Plugin manager instance
        """Access to plugin manager."""
        return self._plugin_manager
    
    @property
    def job_queue(self) -> JobQueue:  # Job queue instance
        """Access to host-owned job queue."""
        return self._job_queue
    
    @property
    def source_service(self) -> SourceService:  # Source service instance
        """Access to source service."""
        return self._source_service
    
    @property
    def graph_service(self) -> GraphService:  # Graph service instance
        """Access to graph service."""
        return self._graph_service
    
    @property
    def verify_service(self) -> VerifyService:  # Verify service instance
        """Access to verify service."""
        return self._verify_service
    
    @property
    def state_store(self) -> SQLiteWorkflowStateStore:  # State store instance
        """Access to state store."""
        return self._state_store
    
    @property
    def routers(self) -> List[APIRouter]:  # List of workflow routers
        """Access to workflow routers (excludes stepflow router)."""
        return self._routers
    
    @property
    def stepflow_router(self) -> APIRouter:  # StepFlow-generated router
        """StepFlow-generated router."""
        return self._stepflow_router

In [ ]:
#| export
@patch
def setup(
    self: StructureDecompWorkflow,
    app  # FastHTML application instance
) -> None:
    """Initialize workflow with FastHTML app."""
    self._app = app

In [ ]:
#| export
@patch
def cleanup(
    self: StructureDecompWorkflow
) -> None:
    """Clean up workflow resources."""
    self._app = None

In [ ]:
#| export
@patch
def get_routers(
    self: StructureDecompWorkflow
) -> List[APIRouter]:  # List of routers to register
    """Return all routers for registration with the app."""
    return self._routers + [self._stepflow_router]

In [ ]:
#| export
@patch
def render_entry_point(
    self: StructureDecompWorkflow,
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # FastHTML component
    """Render the workflow entry point for embedding."""
    # Check if required plugins are available
    sources = self._source_service.get_available_sources()
    
    if not sources:
        # No source plugins available
        content = [
            P(
                "No transcription sources available. Please load transcription plugins first.",
                cls=combine_classes(text_tiers.tertiary, m.b(4))
            )
        ]
        if self.config.no_plugins_redirect:
            content.append(
                A(
                    "Go to Settings",
                    href=self.config.no_plugins_redirect,
                    cls=combine_classes(btn, btn_colors.primary)
                )
            )
        return Div(*content, id=self.config.container_id)
    
    # Sources available - load current status asynchronously
    current_status_url = self._core_routes["current_status"].to()
    return AsyncLoadingContainer(
        container_id=self.config.container_id,
        load_url=current_status_url
    )

## Shared StepFlow Utilities

In [ ]:
#| export
def _create_data_loaders(
    workflow: StructureDecompWorkflow  # Workflow instance for service access
):  # (load_sources, load_empty) callables
    """Create data loader functions for StepFlow steps."""
    def load_sources(request) -> Dict[str, Any]:
        """Load available transcription sources."""
        # Eagerly restore external DB paths so source browser includes them
        selection_result = getattr(workflow, '_selection_result', None)
        if selection_result is not None:
            session_id = get_session_id(request.session)
            selection_result.restore_state(session_id)

        sources = workflow._source_service.get_available_sources()
        transcriptions = workflow._source_service.query_transcriptions(limit=50)
        return {"sources": sources, "transcriptions": transcriptions}
    
    def load_empty(request) -> Dict[str, Any]:
        """No-op loader for steps that load data via init routes."""
        return {}
    
    return load_sources, load_empty

def _validate_always_true(
    state: Dict[str, Any]  # Workflow state dictionary
) -> bool:  # Always True
    """No-op validator for steps that are always valid."""
    return True

## Selection Step

In [ ]:
#| export
def _validate_selection(
    state: Dict[str, Any]  # Workflow state dictionary
) -> bool:  # True if at least one source is selected
    """Validate that sources have been selected."""
    step_states = state.get("step_states", {})
    selection_state = step_states.get("selection", {})
    selected_sources = selection_state.get("selected_sources", [])
    return len(selected_sources) > 0

def _create_selection_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance for URL access
):  # Render callable for StepFlow
    """Create render function for the source selection step."""
    def render(ctx: InteractionContext):
        sources = ctx.get_data("sources", [])
        transcriptions = ctx.get_data("transcriptions", [])
        
        step_state = ctx.state.get("step_states", {}).get("selection", {})
        selected_sources = step_state.get("selected_sources", [])
        grouping_mode = step_state.get("grouping_mode", "media_path")
        
        return render_selection_step(
            sources=sources,
            transcriptions=transcriptions,
            selected_sources=selected_sources,
            grouping_mode=grouping_mode,
            active_tab="db",
            urls=getattr(workflow, '_selection_urls', SelectionUrls()),
            render_local_files_panel=getattr(workflow, '_render_local_files_panel', None),
            sb_state=getattr(workflow, '_sb_state', None),
        )
    return render

## Segmentation Step

In [ ]:
#| export
def _create_combined_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance with _sa_result set
):  # Render callable for StepFlow
    """Create render function for the segmentation & alignment step."""
    def render(ctx: InteractionContext):
        return workflow._sa_result.render_step(ctx)
    return render

## Review Step

In [ ]:
#| export
def _create_review_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance for URL and state access
):  # Render callable for StepFlow
    """Create render function for the review step."""
    def render(ctx: InteractionContext):
        step_states = ctx.state.get("step_states", {})
        
        # Extract segments from segmentation step
        seg_state = step_states.get("segmentation", {})
        segment_dicts = seg_state.get("segments", [])
        segments = [TextSegment.from_dict(s) for s in segment_dicts]
        
        # Extract VAD chunks from alignment step
        align_state = step_states.get("alignment", {})
        chunk_dicts = align_state.get("vad_chunks", [])
        vad_chunks = [VADChunk.from_dict(c) for c in chunk_dicts]
        media_paths = align_state.get("media_paths", [])
        review_urls_obj = getattr(workflow, '_review_urls', ReviewUrls())
        audio_src_url = review_urls_obj.audio_src
        audio_urls = [f"{audio_src_url}?path={mp}" for mp in media_paths] if audio_src_url and media_paths else []
        
        # Assemble segments with their corresponding VAD chunks
        assembled = [
            AssembledSegment(segment=seg, vad_chunk=chunk)
            for seg, chunk in zip(segments, vad_chunks)
        ]
        
        # Extract review step state
        review_state = step_states.get("review", {})
        focused_index = review_state.get("focused_index", 0)
        visible_count = review_state.get("visible_count", 5)
        is_auto_mode = review_state.get("is_auto_mode", False)
        card_width = review_state.get("card_width", 50)
        playback_speed = review_state.get("playback_speed", 1.0)
        auto_navigate = review_state.get("auto_navigate", False)
        
        # Get document title from state, or auto-generate from media path
        media_path = align_state.get("media_path")
        document_title = review_state.get("document_title") or generate_document_title(media_path)
        
        return render_review_step(
            assembled=assembled,
            focused_index=focused_index,
            visible_count=visible_count,
            is_auto_mode=is_auto_mode,
            card_width=card_width,
            playback_speed=playback_speed,
            auto_navigate=auto_navigate,
            document_title=document_title,
            urls=review_urls_obj,
            audio_urls=audio_urls,
        )
    return render

def _create_review_hook(
    workflow: StructureDecompWorkflow  # Workflow instance for service and state access
):  # on_leave callable
    """Create on_leave hook for the review step (commits to graph)."""
    async def on_leave_review(state: Dict[str, Any], request, sess):
        """Commit to graph when leaving Review step."""
        step_states = state.get("step_states", {})
        
        # Extract segments from segmentation step
        seg_state = step_states.get("segmentation", {})
        segment_dicts = seg_state.get("segments", [])
        segments = [TextSegment.from_dict(s) for s in segment_dicts]
        
        # Extract VAD chunks and media path from alignment step
        align_state = step_states.get("alignment", {})
        chunk_dicts = align_state.get("vad_chunks", [])
        vad_chunks = [VADChunk.from_dict(c) for c in chunk_dicts]
        media_path = align_state.get("media_path")
        
        # Get document title from review state, or generate from media path
        review_state = step_states.get("review", {})
        document_title = review_state.get("document_title") or generate_document_title(media_path)
        
        # Check if graph service is available
        if not workflow._graph_service.is_available():
            return Div(
                P("Graph plugin not loaded. Cannot commit document."),
                id=workflow.config.container_id
            )
        
        # Commit to graph
        try:
            result = await workflow._graph_service.commit_document_async(
                title=document_title,
                text_segments=segments,
                vad_chunks=vad_chunks,
                media_type="audio",
            )
            document_id = result.get("document_id")
            
            # Store document_id in review state for verify step to pick up
            session_id = get_session_id(sess)
            workflow_state = workflow._state_store.get_state(workflow.config.workflow_id, session_id)
            step_states = workflow_state.get("step_states", {})
            review_state = step_states.get("review", {})
            review_state["document_id"] = document_id
            review_state["media_path"] = media_path
            step_states["review"] = review_state
            workflow_state["step_states"] = step_states
            workflow._state_store.update_state(workflow.config.workflow_id, session_id, workflow_state)
            
            return None
            
        except Exception as e:
            return Div(
                P(f"Commit failed: {str(e)}"),
                id=workflow.config.container_id
            )
    
    return on_leave_review

## Verify Step

In [ ]:
#| export
def _create_verify_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance for URL access
):  # Render callable for StepFlow
    """Create render function for the verify step."""
    def render(ctx: InteractionContext):
        from cjm_transcript_verify.models import VerificationResult
        
        step_states = ctx.state.get("step_states", {})
        verify_state = step_states.get("verify", {})
        result_dict = verify_state.get("verification_result")
        
        if not result_dict:
            return render_verify_step(
                result=None,
                urls=getattr(workflow, '_verify_urls', VerifyUrls()),
                error="No verification data found. Please complete the review step first."
            )
        
        result = VerificationResult.from_dict(result_dict)
        
        return render_verify_step(
            result=result,
            urls=getattr(workflow, '_verify_urls', VerifyUrls()),
            error=""
        )
    return render

def _create_verify_hooks(
    workflow: StructureDecompWorkflow  # Workflow instance for service and state access
):  # (on_enter_verify, on_complete) callables
    """Create lifecycle hooks for the verify step."""
    async def on_enter_verify(state: Dict[str, Any], request, sess):
        """Query verification results when entering Verify step."""
        step_states = state.get("step_states", {})
        review_state = step_states.get("review", {})
        document_id = review_state.get("document_id")
        
        if not document_id:
            return None
        
        verify_result = await workflow._verify_service.verify_document_async(document_id)
        
        if verify_result:
            session_id = get_session_id(sess)
            workflow_state = workflow._state_store.get_state(workflow.config.workflow_id, session_id)
            step_states = workflow_state.get("step_states", {})
            verify_state = step_states.get("verify", {})
            verify_state["verification_result"] = verify_result.to_dict()
            verify_state["document_id"] = document_id
            step_states["verify"] = verify_state
            workflow_state["step_states"] = step_states
            workflow._state_store.update_state(workflow.config.workflow_id, session_id, workflow_state)
        
        return None
    
    async def on_complete(state: Dict[str, Any], request):
        """Handle workflow completion.
        
        If `workflow.on_complete_redirect_url` is set (e.g. by a host wiring up
        session management), perform a full page navigation to that URL via
        JavaScript. Otherwise fall back to the original reset-in-place behavior
        via HTMX. The redirect URL is read dynamically from the workflow instance
        so hosts can set it any time before the user clicks "Start New Workflow".
        
        A JS redirect (not an HTMX swap) is used because the target page is a
        completely different layout — a full page navigation ensures the target
        route's `handle_htmx_request` applies the full layout wrapper (navbar, etc.).
        """
        redirect_url = workflow._on_complete_redirect_url
        if redirect_url:
            return Div(
                Script(f"window.location.href = '{redirect_url}';"),
                id=workflow.config.container_id,
            )
        return Div(
            P("Workflow complete. Starting new workflow..."),
            hx_get=workflow._stepflow_router.reset.to(),
            hx_trigger="load",
            hx_target=f"#{workflow.config.container_id}",
            hx_swap="outerHTML",
            id=workflow.config.container_id,
        )
    
    return on_enter_verify, on_complete

In [ ]:
#| export
@patch
def _create_step_flow(
    self: StructureDecompWorkflow
) -> StepFlow:  # Configured StepFlow instance
    """Create and configure the StepFlow instance."""
    workflow = self
    
    load_sources, load_empty = _create_data_loaders(workflow)
    on_leave_review = _create_review_hook(workflow)
    on_enter_verify, on_complete = _create_verify_hooks(workflow)
    
    # Build step progress renderer (maps Step objects to StepInfo for the progress bar)
    def _progress_renderer(steps, current_index):
        return render_step_progress(
            [StepInfo(s.title) for s in steps], current_index
        )
    
    return StepFlow(
        debug=True,
        flow_id=self.config.workflow_id,
        state_store=self._session_adapter,
        container_id=self.config.container_id,
        steps=[
            Step(
                id="selection",
                title="Select Sources",
                render=_create_selection_renderer(workflow),
                validate=_validate_selection,
                data_loader=load_sources,
                data_keys=["selected_sources"],
                show_back=False,
                show_cancel=True,
                next_button_text="Continue"
            ),
            Step(
                id="segmentation",
                title="Segment & Align",
                render=_create_combined_renderer(workflow),
                validate=workflow._sa_result.validate_alignment,
                data_loader=load_empty,
                data_keys=["segments"],
                show_back=True,
                show_cancel=True,
                next_button_text="Continue"
            ),
            Step(
                id="review",
                title="Review",
                render=_create_review_renderer(workflow),
                validate=_validate_always_true,
                data_loader=load_empty,
                data_keys=[],
                show_back=True,
                show_cancel=True,
                next_button_text="Commit to Graph",
                on_leave=on_leave_review
            ),
            Step(
                id="verify",
                title="Verify",
                render=_create_verify_renderer(workflow),
                validate=_validate_always_true,
                data_loader=load_empty,
                data_keys=[],
                show_back=True,
                show_cancel=False,
                next_button_text="Start New Workflow",
                on_enter=on_enter_verify
            )
        ],
        on_complete=on_complete,
        show_progress=self.config.show_progress,
        progress_renderer=_progress_renderer,
        wrap_in_form=True
    )

In [ ]:
#| export
@patch
def _create_routers(
    self: StructureDecompWorkflow
) -> List[APIRouter]:  # List of configured APIRouters
    """Create the workflow's API routers."""
    from cjm_fasthtml_workflow_transcript_decomp.routes.init import init_routers
    return init_routers(self)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()